# 🔥 DWSIM - Optimize Natural Gas Processing

> **Created by:**
> Prof. Roymel Rodríguez Carpio, Ph.D.
> Prof. Nicolas Spogis, Ph.D.
> Gabriel Freitas

---

## Import the Libraries

### Libraries for array and dataset manipulation

In [1]:
import numpy as np
import pandas as pd
import time

### Libraries for plotting results

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns


### Set NumPy Seed

In [3]:
seed = 42
np.random.seed(seed)

## Import the Specific Libraries to run DWSIM

In [4]:
# remove the following two lines to run on linux
import pythoncom  # Component of the pywin32 library used to interact with COM (Component Object Model)

pythoncom.CoInitialize()

In [5]:
import clr  # Pythonnet

## Change your DWSIM installation path and load dlls

In [6]:
dwsimpath = r"C:\Users\nicol\AppData\Local\DWSIM"

In [7]:
clr.AddReference(dwsimpath + "\\CapeOpen.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "\\DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "\\DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "\\DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "\\System.Buffers.dll")
clr.AddReference(dwsimpath + "\\DWSIM.Thermodynamics.ThermoC.dll")



## Open DWSIM Automation def

In [8]:
def open_DWSIM(FlowsheetFile):
    from DWSIM.Automation import Automation3
    Manager = Automation3()
    Flowsheet = Manager.LoadFlowsheet(FlowsheetFile)
    return Manager, Flowsheet

## Read and Save Snapshot defs

In [9]:
def SaveSnapshot(dwsimpath, Flowsheet):
    clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
    from DWSIM.Interfaces.Enums import SnapshotType
    Type = SnapshotType.ObjectData
    snap = Flowsheet.GetSnapshot(Type)
    return snap


def ReadSnapshot(dwsimpath, snap, Flowsheet):
    clr.AddReference(dwsimpath + "\\DWSIM.Interfaces.dll")
    from DWSIM.Interfaces.Enums import SnapshotType
    Type = SnapshotType.ObjectData
    Flowsheet.RestoreSnapshot(snap, Type)
    return snap

## Open DWSIM File and load initial Snapshot

In [10]:
FlowsheetFileName = "DWSIM/NaturalGasProcessing.dwxmz"
Manager, Flowsheet = open_DWSIM(FlowsheetFileName)

In [11]:
Snapshot = SaveSnapshot(dwsimpath, Flowsheet)

## Call DWSIM -> Change Input Parameters -> Solve -> Get Output Parameters

In [12]:
def run_dwsim_simulation(u):
    global Manager, Flowsheet, Snapshot, dwsimpath

    # Read Snapshot
    ReadSnapshot(dwsimpath, Snapshot, Flowsheet)

    # Extract decision variables
    V_123102_Temp = u[0]
    T_123701_Reboiler = u[1]
    T_123702_Condenser = u[2]
    T_123702_Reboiler = u[3]
    T_123703_Reboiler = u[4]

    # Set Spreadsheet Object
    mySpreadsheet = Flowsheet.GetSpreadsheetObject()
    ws = mySpreadsheet.Worksheets["MAIN"]

    try:
        # Set Input Parameters
        ws.Cells["B2"].Data = V_123102_Temp
        ws.Cells["B3"].Data = T_123701_Reboiler
        ws.Cells["B4"].Data = T_123702_Condenser
        ws.Cells["B5"].Data = T_123702_Reboiler
        ws.Cells["B6"].Data = T_123703_Reboiler

        # Update Input Spreadsheet
        ws.Recalculate()

        # Request a calculation
        Manager.CalculateFlowsheet4(Flowsheet)

        # Check if the Flowsheet was Calculated Correctly
        if Flowsheet.Solved == False:
            # Penalty for failed simulation
            return np.array([1e6, 1e6, 1e6, 1e6, 1e6,
                         1e6, 1e6, 1e6, 1e6, 1e6, 1e6])

        # Update Output Spreadsheet
        ws.Recalculate()

        # Get Output Parameters
        Sale_Price = float(ws.Cells["B17"].Data)
        GVC1 = float(ws.Cells["B10"].Data)
        GVC2 = float(ws.Cells["B12"].Data)
        GVC5 = float(ws.Cells["B15"].Data)
        GVC5_PVR = float(ws.Cells["B16"].Data)

        # Get Auxiliary Output Parameters
        T_123701_Stage1_Temperature = float(ws.Cells["B19"].Data)
        T_123701_Stage11_Temperature = float(ws.Cells["B20"].Data)
        T_123702_Stage1_Temperature = float(ws.Cells["B21"].Data)
        T_123702_Stage21_Temperature = float(ws.Cells["B22"].Data)
        T_123703_Stage1_Temperature = float(ws.Cells["B23"].Data)
        T_123703_Stage6_Temperature = float(ws.Cells["B24"].Data)

        # ========================================
        # Objective Function: Maximize Sales Price
        # ========================================
        f_obj = -Sale_Price  # we want to maximize this

        # ========================================
        # Constraints
        # ========================================
        g1 = GVC1  # Should be >= 80 %
        g2 = GVC2  # Should be <=12 %
        g3 = GVC5  # Should be <=2 %
        g4 = GVC5_PVR  # Should be <= 76 kPa

        return np.array([f_obj, g1, g2, g3, g4,
                         T_123701_Stage1_Temperature,
                         T_123701_Stage11_Temperature,
                         T_123702_Stage1_Temperature,
                         T_123702_Stage21_Temperature,
                         T_123703_Stage1_Temperature,
                         T_123703_Stage6_Temperature,])

    except Exception as e:
        print(f"Error in simulation: {e}")
        return np.array([1e6, 1e6, 1e6, 1e6, 1e6,
                         1e6, 1e6, 1e6, 1e6, 1e6, 1e6])



## Variable bounds

In [13]:
# Decision variable bounds
# [V_123102_Temp, T_123701_Reboiler, T_123702_Condenser, T_123702_Reboiler, T_123703_Reboiler]
LB = np.array([-33.0, 60.0, 0.7,  9.0, 45.0])  # Lower bounds
UB = np.array([-17.0, 77.0, 4.5, 27.0, 64.0])  # Upper bounds

bounds = list(zip(LB, UB))

print("\nDecision Variable Bounds:")
print(f"  V_123102_Temp: [{LB[0]:.0f}, {UB[0]:.0f}]")
print(f"  T_123701_Reboiler: [{LB[1]:.0f}, {UB[1]:.0f}]")
print(f"  T_123702_Condenser: [{LB[2]:.0f}, {UB[2]:.0f}]")
print(f"  T_123702_Reboiler: [{LB[3]:.0f}, {UB[3]:.0f}]")
print(f"  T_123703_Reboiler: [{LB[4]:.0f}, {UB[4]:.0f}]")



Decision Variable Bounds:
  V_123102_Temp: [-33, -17]
  T_123701_Reboiler: [60, 77]
  T_123702_Condenser: [1, 4]
  T_123702_Reboiler: [9, 27]
  T_123703_Reboiler: [45, 64]


## Test simulation with initial values

In [14]:
print("=" * 60)
print("TESTING DWSIM SIMULATION - Lower bounds")
print("=" * 60)

(f_obj, g1, g2, g3, g4,
 T_123701_Stage1_Temperature,
 T_123701_Stage11_Temperature,
 T_123702_Stage1_Temperature,
 T_123702_Stage21_Temperature,
 T_123703_Stage1_Temperature,
 T_123703_Stage6_Temperature) = run_dwsim_simulation(LB)

print(f"\nTest Input:")
print(f"  V_123102_Temp = {LB[0]:.2f}")
print(f"  T_123701_Reboiler = {LB[1]:.2f}")
print(f"  T_123702_Condenser = {LB[2]:.2f}")
print(f"  T_123702_Reboiler = {LB[3]:.2f}")
print(f"  T_123703_Reboiler = {LB[4]:.2f}")

print(f"\nTest Output:")
print(f"  Objective (Sales Price) = {-f_obj:.2f} US$")
print(f"  g1 (GV.C1 [%]) = {g1:.2f} (should be >= 80 %)")
print(f"  g2 (GLP.C2 [%]) = {g2:.2f} (should be <=12 %)")
print(f"  g3 (GLP.C5 [%]) = {g3:.2f} (should be <=2 %)")
print(f"  g4 (C5+.PVR [kPa]) = {g4:.2f} (should be <= 76 kPa)")

print(f"\nTemperatures Output:")
print(f"  T 123701 Stage 1 Temperature = {T_123701_Stage1_Temperature:.2f} ºC")
print(f"  T 123701 Stage 11 Temperature = {T_123701_Stage11_Temperature:.2f} ºC")
print(f"  T 123702 Stage 1 Temperature = {T_123702_Stage1_Temperature:.2f} ºC")
print(f"  T 123702 Stage 21 Temperature = {T_123702_Stage21_Temperature:.2f} ºC")
print(f"  T 123703 Stage 1 Temperature = {T_123703_Stage1_Temperature:.2f} ºC")
print(f"  T 123703 Stage 6 Temperature = {T_123703_Stage6_Temperature:.2f} ºC")

TESTING DWSIM SIMULATION - Lower bounds

Test Input:
  V_123102_Temp = -33.00
  T_123701_Reboiler = 60.00
  T_123702_Condenser = 0.70
  T_123702_Reboiler = 9.00
  T_123703_Reboiler = 45.00

Test Output:
  Objective (Sales Price) = 658.72 US$
  g1 (GV.C1 [%]) = 83.54 (should be >= 80 %)
  g2 (GLP.C2 [%]) = 8.08 (should be <=12 %)
  g3 (GLP.C5 [%]) = 6.71 (should be <=2 %)
  g4 (C5+.PVR [kPa]) = 32.65 (should be <= 76 kPa)

Temperatures Output:
  T 123701 Stage 1 Temperature = 16.89 ºC
  T 123701 Stage 11 Temperature = 71.36 ºC
  T 123702 Stage 1 Temperature = 42.77 ºC
  T 123702 Stage 21 Temperature = 166.33 ºC
  T 123703 Stage 1 Temperature = 111.86 ºC
  T 123703 Stage 6 Temperature = 205.59 ºC


In [15]:
print("=" * 60)
print("TESTING DWSIM SIMULATION - Upper bounds")
print("=" * 60)

(f_obj, g1, g2, g3, g4,
 T_123701_Stage1_Temperature,
 T_123701_Stage11_Temperature,
 T_123702_Stage1_Temperature,
 T_123702_Stage21_Temperature,
 T_123703_Stage1_Temperature,
 T_123703_Stage6_Temperature) = run_dwsim_simulation(UB)


print(f"\nTest Input:")
print(f"  V_123102_Temp = {UB[0]:.2f}")
print(f"  T_123701_Reboiler = {UB[1]:.2f}")
print(f"  T_123702_Condenser = {UB[2]:.2f}")
print(f"  T_123702_Reboiler = {UB[3]:.2f}")
print(f"  T_123703_Reboiler = {UB[4]:.2f}")

print(f"\nTest Output:")
print(f"  Objective (Sales Price) = {-f_obj:.2f} US$")
print(f"  g1 (GV.C1 [%]) = {g1:.2f} (should be >= 80 %)")
print(f"  g2 (GLP.C2 [%]) = {g2:.2f} (should be <=12 %)")
print(f"  g3 (GLP.C5 [%]) = {g3:.2f} (should be <=2 %)")
print(f"  g4 (C5+.PVR [kPa]) = {g4:.2f} (should be <= 76 kPa)")

print(f"\nTemperatures Output:")
print(f"  T 123701 Stage 1 Temperature = {T_123701_Stage1_Temperature:.2f} ºC")
print(f"  T 123701 Stage 11 Temperature = {T_123701_Stage11_Temperature:.2f} ºC")
print(f"  T 123702 Stage 1 Temperature = {T_123702_Stage1_Temperature:.2f} ºC")
print(f"  T 123702 Stage 21 Temperature = {T_123702_Stage21_Temperature:.2f} ºC")
print(f"  T 123703 Stage 1 Temperature = {T_123703_Stage1_Temperature:.2f} ºC")
print(f"  T 123703 Stage 6 Temperature = {T_123703_Stage6_Temperature:.2f} ºC")

TESTING DWSIM SIMULATION - Upper bounds

Test Input:
  V_123102_Temp = -17.00
  T_123701_Reboiler = 77.00
  T_123702_Condenser = 4.50
  T_123702_Reboiler = 27.00
  T_123703_Reboiler = 64.00

Test Output:
  Objective (Sales Price) = 626.16 US$
  g1 (GV.C1 [%]) = 81.13 (should be >= 80 %)
  g2 (GLP.C2 [%]) = 14.28 (should be <=12 %)
  g3 (GLP.C5 [%]) = 0.00 (should be <=2 %)
  g4 (C5+.PVR [kPa]) = 117.01 (should be <= 76 kPa)

Temperatures Output:
  T 123701 Stage 1 Temperature = 14.40 ºC
  T 123701 Stage 11 Temperature = 69.11 ºC
  T 123702 Stage 1 Temperature = 30.74 ºC
  T 123702 Stage 21 Temperature = 120.65 ºC
  T 123703 Stage 1 Temperature = 91.23 ºC
  T 123703 Stage 6 Temperature = 151.74 ºC


## Class Evaluator to avoid recomputation

In [16]:
class Evaluator:
    def __init__(self):
        self.cache = {}
        self.n_evaluations = 0

    def evaluate(self, x):
        key = tuple(np.round(x, 6))  # Less precision for caching
        if key not in self.cache:
            self.cache[key] = run_dwsim_simulation(x)
            self.n_evaluations += 1
        return self.cache[key]

    def objective(self, x):
        return self.evaluate(x)[0]

    def g1(self, x):
        return self.evaluate(x)[1]

    def g2(self, x):
        return self.evaluate(x)[2]

    def g3(self, x):
        return self.evaluate(x)[3]

    def g4(self, x):
        return self.evaluate(x)[4]

    def reset_counter(self):
        self.n_evaluations = 0

## Create global evaluator

In [17]:
ev = Evaluator()

## Constraint checking function

In [18]:
def check_constraints_scipy(x, constraints, tol=1e-6):
    """Check constraints for scipy optimization"""
    report = []

    for i, c in enumerate(constraints, 1):
        val = np.atleast_1d(c.fun(x))
        lb = np.atleast_1d(c.lb)
        ub = np.atleast_1d(c.ub)

        viol_low = np.maximum(lb - val, 0.0)
        viol_high = np.maximum(val - ub, 0.0)
        violation = np.maximum(viol_low, viol_high)

        ok = np.all(violation <= tol)

        report.append({
            "id": i,
            "ok": ok,
            "value": val,
            "lb": lb,
            "ub": ub,
            "violation": violation
        })

    return report

# ========================================
# Create LHS Dataset
# ========================================

In [19]:
from scipy.stats import qmc, norm
from tqdm import tqdm

In [20]:
# Bounds array: each row is [min, max] for each variable
LHS_bounds = np.column_stack((LB, UB))

In [21]:
n_samples = 2000
n_variables = LHS_bounds.shape[0]

In [22]:
# =============================================================================
# Generate LHS samples
# =============================================================================
print("Generating Latin Hypercube Samples...")

# Create LHS sampler with seed for reproducibility
sampler = qmc.LatinHypercube(d=n_variables, seed=42)

# Generate samples in [0, 1] range
lhs_samples_unit = sampler.random(n=n_samples)

# Scale samples to actual bounds
lhs_samples = qmc.scale(lhs_samples_unit, LHS_bounds[:, 0], LHS_bounds[:, 1])

print(f"Generated {n_samples} LHS samples successfully!")

Generating Latin Hypercube Samples...
Generated 2000 LHS samples successfully!


In [23]:
# =============================================================================
# Run DWSIM simulations for all samples
# =============================================================================
print(f"\nRunning DWSIM simulations for {n_samples} samples...")
print("This may take a while...\n")

# Initialize output arrays
outputs = np.zeros((n_samples, 11))

# Run simulations with progress bar
for i, sample in enumerate(tqdm(lhs_samples, desc="Simulating", unit="sample")):
    outputs[i, :] = run_dwsim_simulation(sample)

print("\nSimulations completed!")


Running DWSIM simulations for 2000 samples...
This may take a while...



Simulating: 100%|██████████| 2000/2000 [2:08:20<00:00,  3.85s/sample]  


Simulations completed!


In [24]:
# =============================================================================
# Create DataFrame and save dataset
# =============================================================================
# Column names
input_columns = ['V_123102_Temp',
                 'T_123701_Reboiler',
                 'T_123702_Condenser',
                 'T_123702_Reboiler',
                 'T_123703_Reboiler']

output_columns = ['Sales_Price', 'GVC1', 'GVC2', 'GVC5', 'GVC5_PVR',
                  'T_123701_Stage1_Temperature',
                  'T_123701_Stage11_Temperature',
                  'T_123702_Stage1_Temperature',
                  'T_123702_Stage21_Temperature',
                  'T_123703_Stage1_Temperature',
                  'T_123703_Stage6_Temperature']

# Create DataFrame
df = pd.DataFrame(
    data=np.hstack([lhs_samples, outputs]),
    columns=input_columns + output_columns
)

# Count valid and invalid samples
mask_invalid = (df[output_columns] == 1e6).all(axis=1)
df_clean = df[~mask_invalid]
n_valid = df_clean.shape[0]
n_invalid = n_samples - n_valid

print(f"\n{'='*60}")
print(f"Dataset Summary")
print(f"{'='*60}")
print(f"Total samples:   {n_samples}")
print(f"Valid samples:   {n_valid} ({100*n_valid/n_samples:.1f}%)")
print(f"Invalid samples: {n_invalid} ({100*n_invalid/n_samples:.1f}%)")
print(f"{'='*60}")


Dataset Summary
Total samples:   2000
Valid samples:   2000 (100.0%)
Invalid samples: 0 (0.0%)


In [25]:
# Display statistics
print("\nDataset Statistics:")
print(df.describe().round(4))


Dataset Statistics:
       V_123102_Temp  T_123701_Reboiler  T_123702_Condenser  \
count      2000.0000          2000.0000           2000.0000   
mean        -24.9999            68.5000              2.6000   
std           4.6200             4.9087              1.0973   
min         -33.0000            60.0038              0.7011   
25%         -28.9971            64.2485              1.6501   
50%         -24.9984            68.5027              2.6007   
75%         -21.0046            72.7450              3.5500   
max         -17.0031            76.9983              4.4994   

       T_123702_Reboiler  T_123703_Reboiler  Sales_Price       GVC1  \
count          2000.0000          2000.0000    2000.0000  2000.0000   
mean             18.0000            54.5000    -644.5029    82.4302   
std               5.1974             5.4862      13.8967     0.9233   
min               9.0069            45.0044    -673.3515    80.7304   
25%              13.5050            49.7551    -656.4881

In [26]:
# =============================================================================
# Save dataset to CSV
# =============================================================================
# Save full dataset
output_file_full = "datasets/dataset_full.csv"
df.to_csv(output_file_full, index=False)
print(f"\nFull dataset saved to: {output_file_full}")

# Save clean dataset
output_file_clean = "datasets/dataset_clean.csv"
df_clean.to_csv(output_file_clean, index=False)
print(f"Clean dataset saved to: {output_file_clean}")

print(f"\n{'='*60}")
print("Dataset generation completed successfully!")
print(f"{'='*60}")


Full dataset saved to: datasets/dataset_full.csv
Clean dataset saved to: datasets/dataset_clean.csv

Dataset generation completed successfully!


# ========================================
# LOCAL OPTIMIZATION: SLSQP (Scipy - Baseline)
# ========================================

In [27]:
# ============================================================
# Load LHS dataset
# ============================================================
df = pd.read_csv("datasets/dataset_clean.csv")

# Column names
decision_vars = [
    'V_123102_Temp',
    'T_123701_Reboiler',
    'T_123702_Condenser',
    'T_123702_Reboiler',
    'T_123703_Reboiler',]

obj_col = 'Sales_Price'
g1_col = 'GVC1'       # >= 80
g2_col = 'GVC2'       # <= 12
g3_col = 'GVC5'       # <= 2
g4_col = 'GVC5_PVR'   # <= 76

# ============================================================
# Define constraint limits
# ============================================================
constraints_limits = {
    g1_col: {'lb': 80.0, 'ub': np.inf},     # g1 >= 80
    g2_col: {'lb': 0.0, 'ub': 12.0},        # g2 <= 12
    g3_col: {'lb': 0.0, 'ub': 2.0},         # g3 <= 2
    g4_col: {'lb': 0.0, 'ub': 76.0},        # g4 <= 76
}

# ============================================================
# Check feasibility
# ============================================================
feasible_mask = pd.Series(True, index=df.index)

for col, lim in constraints_limits.items():
    if np.isfinite(lim['lb']):
        feasible_mask &= (df[col] >= lim['lb'])
    if np.isfinite(lim['ub']):
        feasible_mask &= (df[col] <= lim['ub'])

df_feasible = df[feasible_mask]
n_total = len(df)
n_feasible = len(df_feasible)

print(f"Total LHS samples: {n_total}")
print(f"Feasible samples:  {n_feasible} ({100*n_feasible/n_total:.1f}%)")

# ============================================================
# Select best x0
# ============================================================
if n_feasible > 0:
    # Best feasible point (min Sales_Price = most negative = highest real price)
    best_idx = df_feasible[obj_col].idxmin()
    best_row = df_feasible.loc[best_idx]
    print(f"\n--- Best feasible point (index {best_idx}) ---")
else:
    # No feasible point: pick the one with minimum total violation
    print("\nNo fully feasible point found. Selecting least-violated point...")

    violation = pd.Series(0.0, index=df.index)
    for col, lim in constraints_limits.items():
        if np.isfinite(lim['lb']):
            violation += np.maximum(lim['lb'] - df[col], 0.0)
        if np.isfinite(lim['ub']):
            violation += np.maximum(df[col] - lim['ub'], 0.0)

    best_idx = violation.idxmin()
    best_row = df.loc[best_idx]
    print(f"\n--- Least-violated point (index {best_idx}, "
          f"total violation = {violation[best_idx]:.4f}) ---")

# Extract x0
x0 = best_row[decision_vars].values.astype(float)

print(f"\nObjective (Sales_Price): {best_row[obj_col]:.4f}")
print(f"Real Price: {-best_row[obj_col]:.4f}")
print(f"\nDecision variables (x0):")
for name, val in zip(decision_vars, x0):
    print(f"  {name} = {val:.4f}")

print(f"\nConstraint values:")
for col, lim in constraints_limits.items():
    val = best_row[col]
    lb_str = f"{lim['lb']:.1f}" if np.isfinite(lim['lb']) else "-inf"
    ub_str = f"{lim['ub']:.1f}" if np.isfinite(lim['ub']) else "inf"
    ok = True
    if np.isfinite(lim['lb']) and val < lim['lb']:
        ok = False
    if np.isfinite(lim['ub']) and val > lim['ub']:
        ok = False
    status = "✓" if ok else "✗"
    print(f"  {col} = {val:.4f}  [{lb_str}, {ub_str}]  {status}")

# ============================================================
# x0 is ready to use in your optimization
# ============================================================
print(f"\nx0 = np.array({list(x0)})")

Total LHS samples: 2000
Feasible samples:  281 (14.1%)

--- Best feasible point (index 1591) ---

Objective (Sales_Price): -662.8284
Real Price: 662.8284

Decision variables (x0):
  V_123102_Temp = -32.8294
  T_123701_Reboiler = 64.0769
  T_123702_Condenser = 4.2981
  T_123702_Reboiler = 13.8861
  T_123703_Reboiler = 47.4421

Constraint values:
  GVC1 = 83.7331  [80.0, inf]  ✓
  GVC2 = 11.9135  [0.0, 12.0]  ✓
  GVC5 = 0.1878  [0.0, 2.0]  ✓
  GVC5_PVR = 53.5710  [0.0, 76.0]  ✓

x0 = np.array([-32.82942291491444, 64.07694006124437, 4.298112932449205, 13.886056746901064, 47.442059795382654])


In [28]:
from scipy.optimize import minimize, NonlinearConstraint

print("\n\n" + "=" * 60)
print("OPTIMIZATION 1: SLSQP (SciPy - Local Optimizer)")
print("=" * 60)

# Reset evaluator
ev.reset_counter()
ev.cache.clear()

# Constraints for SLSQP
constraints_scipy = [
    NonlinearConstraint(ev.g1, 80.0, np.inf),       # should be >= 80
    NonlinearConstraint(ev.g2, 0.0, 12.0),          # should be <=12 %
    NonlinearConstraint(ev.g3, 0.0, 2.0),           # should be <=2 %
    NonlinearConstraint(ev.g4, 0.0, 76.0),          # should be <= 76 kPa
]

# Callback
n_iter_slsqp = 0


def callback_slsqp(xk):
    global n_iter_slsqp
    n_iter_slsqp += 1
    print(f"Iter {n_iter_slsqp} | Evaluations {ev.n_evaluations}")


# Options
method = 'SLSQP'
options = {
    'maxiter': 100,
    'ftol': 1e-2,
    'disp': True,
    'eps': 0.5
}

# Run optimization
start_time = time.time()
res_slsqp = minimize(
    ev.objective,
    x0,
    method=method,
    bounds=bounds,
    constraints=constraints_scipy,
    options=options,
    callback=callback_slsqp
)
time_slsqp = time.time() - start_time



OPTIMIZATION 1: SLSQP (SciPy - Local Optimizer)
Iter 1 | Evaluations 9
Iter 2 | Evaluations 15
Iter 3 | Evaluations 21
Iter 4 | Evaluations 27
Iter 5 | Evaluations 32
Optimization terminated successfully    (Exit mode 0)
            Current function value: -663.8537091855615
            Iterations: 5
            Function evaluations: 30
            Gradient evaluations: 5


In [29]:
# Results
print("\n" + "=" * 60)
print("SLSQP RESULTS:")
print("=" * 60)
print(f"Converged: {res_slsqp.success}")
print(f"Message: {res_slsqp.message}")
print(f"\nObjective Value:")
print(f"  Sales Price = {-res_slsqp.fun:.2f}")

print(f"\nOptimal Solution:")
print(f"  V_123102_Temp = {res_slsqp.x[0]:.2f}")
print(f"  T_123701_Reboiler = {res_slsqp.x[1]:.2f}")
print(f"  T_123702_Condenser = {res_slsqp.x[2]:.2f}")
print(f"  T_123702_Reboiler = {res_slsqp.x[3]:.2f}")
print(f"  T_123703_Reboiler = {res_slsqp.x[4]:.2f}")


print(f"\nPerformance:")
print(f"  Iterations: {n_iter_slsqp}")
print(f"  DWSIM evaluations: {ev.n_evaluations}")
print(f"  Optimization time: {time_slsqp:.2f} seconds")

# Check constraints
report = check_constraints_scipy(res_slsqp.x, constraints_scipy)
print("\nConstraint Check:")
for r in report:
    status = "✓ OK" if r['ok'] else "✗ VIOLATED"
    print(f"  Constraint {r['id']}: {status}")
    print(f"    value = {r['value'][0]:.4f}, bounds = [{r['lb'][0]:.2f}, {r['ub'][0]:.2f}]")



SLSQP RESULTS:
Converged: True
Message: Optimization terminated successfully

Objective Value:
  Sales Price = 663.85

Optimal Solution:
  V_123102_Temp = -33.00
  T_123701_Reboiler = 63.53
  T_123702_Condenser = 4.31
  T_123702_Reboiler = 15.96
  T_123703_Reboiler = 48.00

Performance:
  Iterations: 5
  DWSIM evaluations: 32
  Optimization time: 229.80 seconds

Constraint Check:
  Constraint 1: ✓ OK
    value = 83.7441, bounds = [80.00, inf]
  Constraint 2: ✓ OK
    value = 11.9999, bounds = [0.00, 12.00]
  Constraint 3: ✓ OK
    value = 0.0056, bounds = [0.00, 2.00]
  Constraint 4: ✗ VIOLATED
    value = 76.0021, bounds = [0.00, 76.00]


## Run Optimal Solution

In [30]:
Optimal = res_slsqp.x

In [31]:
print(Optimal)

[-33.          63.52815876   4.31232576  15.95793763  47.9998889 ]


In [32]:
print("=" * 60)
print("Optimal Solution")
print("=" * 60)

(f_obj, g1, g2, g3, g4,
 T_123701_Stage1_Temperature,
 T_123701_Stage11_Temperature,
 T_123702_Stage1_Temperature,
 T_123702_Stage21_Temperature,
 T_123703_Stage1_Temperature,
 T_123703_Stage6_Temperature) = run_dwsim_simulation(Optimal)

print(f"\nTest Input:")
print(f"  V_123102_Temp = {Optimal[0]:.2f}")
print(f"  T_123701_Reboiler = {Optimal[1]:.2f}")
print(f"  T_123702_Condenser = {Optimal[2]:.2f}")
print(f"  T_123702_Reboiler = {Optimal[3]:.2f}")
print(f"  T_123703_Reboiler = {Optimal[4]:.2f}")

print(f"\nTest Output:")
print(f"  Objective (Sales Price) = {-f_obj:.2f} US$")
print(f"  g1 (GV.C1 [%]) = {g1:.2f} (should be >= 80 %)")
print(f"  g2 (GLP.C2 [%]) = {g2:.2f} (should be <=12 %)")
print(f"  g3 (GLP.C5 [%]) = {g3:.2f} (should be <=2 %)")
print(f"  g4 (C5+.PVR [kPa]) = {g4:.2f} (should be <= 76 kPa)")

print(f"\nTemperatures Output:")
print(f"  T 123701 Stage 1 Temperature = {T_123701_Stage1_Temperature:.2f} ºC")
print(f"  T 123701 Stage 11 Temperature = {T_123701_Stage11_Temperature:.2f} ºC")
print(f"  T 123702 Stage 1 Temperature = {T_123702_Stage1_Temperature:.2f} ºC")
print(f"  T 123702 Stage 21 Temperature = {T_123702_Stage21_Temperature:.2f} ºC")
print(f"  T 123703 Stage 1 Temperature = {T_123703_Stage1_Temperature:.2f} ºC")
print(f"  T 123703 Stage 6 Temperature = {T_123703_Stage6_Temperature:.2f} ºC")

Optimal Solution

Test Input:
  V_123102_Temp = -33.00
  T_123701_Reboiler = 63.53
  T_123702_Condenser = 4.31
  T_123702_Reboiler = 15.96
  T_123703_Reboiler = 48.00

Test Output:
  Objective (Sales Price) = 663.85 US$
  g1 (GV.C1 [%]) = 83.74 (should be >= 80 %)
  g2 (GLP.C2 [%]) = 12.00 (should be <=12 %)
  g3 (GLP.C5 [%]) = 0.01 (should be <=2 %)
  g4 (C5+.PVR [kPa]) = 76.00 (should be <= 76 kPa)

Temperatures Output:
  T 123701 Stage 1 Temperature = 15.73 ºC
  T 123701 Stage 11 Temperature = 66.29 ºC
  T 123702 Stage 1 Temperature = 34.29 ºC
  T 123702 Stage 21 Temperature = 140.11 ºC
  T 123703 Stage 1 Temperature = 107.44 ºC
  T 123703 Stage 6 Temperature = 197.24 ºC


## ROBUST OPTIMIZATION

In [35]:
import os
FIGDIR = 'operability/optimization_figures'
os.makedirs(FIGDIR, exist_ok=True)

In [36]:
# ============================================================
# CONFIGURATION
# ============================================================

# Revenue from the original SLSQP optimal (constraint-boundary solution)
REV_OPTIMAL = -f_obj  # $/h

# Constraint limits (ANP regulations)
LIM_G1 = 80.0   # SG_C1 >= 80 mol%
LIM_G2 = 12.0   # LPG_C2 <= 12 mol%
LIM_G3 = 2.0    # LPG_C5 <= 2 mol%
LIM_G4 = 76.0   # NG_RVP <= 76 kPa



In [37]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def compute_margins(ev, x):
    """Compute normalised constraint margins (0 = at boundary, 1 = far away)."""
    g1 = ev.g1(x)
    g2 = ev.g2(x)
    g3 = ev.g3(x)
    g4 = ev.g4(x)

    margin_g1 = (g1 - LIM_G1) / LIM_G1        # >= 0 when feasible
    margin_g2 = (LIM_G2 - g2) / LIM_G2         # >= 0 when feasible
    margin_g3 = (LIM_G3 - g3) / LIM_G3         # >= 0 when feasible
    margin_g4 = (LIM_G4 - g4) / LIM_G4         # >= 0 when feasible

    return {
        "SG_C1":  {"value": g1, "margin": margin_g1},
        "LPG_C2": {"value": g2, "margin": margin_g2},
        "LPG_C5": {"value": g3, "margin": margin_g3},
        "NG_RVP": {"value": g4, "margin": margin_g4},
    }


def print_solution(label, x, ev, elapsed=None):
    """Pretty-print an optimization solution."""
    rev = -ev.objective(x)
    margins = compute_margins(ev, x)
    min_margin = min(m["margin"] for m in margins.values())

    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  Revenue = {rev:.2f} $/h")
    print(f"  Min normalised margin = {min_margin:.4f} ({min_margin*100:.2f}%)")
    if elapsed:
        print(f"  Elapsed = {elapsed:.1f} s")

    print(f"\n  Decision variables:")
    var_names = ["V_02_Temp", "T_01_Reb", "T_02_RR", "T_02_Reb", "T_03_Reb"]
    for name, val in zip(var_names, x):
        print(f"    {name:<14} = {val:>10.4f}")

    print(f"\n  Constraints:")
    print(f"    {'Constraint':<12} {'Value':>10} {'Limit':>10} {'Margin':>10} {'Status'}")
    print(f"    {'-'*56}")
    for name, info in margins.items():
        status = "✓" if info["margin"] >= 0 else "✗ VIOLATED"
        margin_pct = f"{info['margin']*100:+.2f}%"
        lim = {"SG_C1": f">= {LIM_G1}", "LPG_C2": f"<= {LIM_G2}",
               "LPG_C5": f"<= {LIM_G3}", "NG_RVP": f"<= {LIM_G4}"}[name]
        print(f"    {name:<12} {info['value']:>10.4f} {lim:>10} {margin_pct:>10} {status}")

    return rev, min_margin




In [38]:
# ============================================================
# APPROACH 1: BACK-OFF (tighten constraint limits)
# ============================================================

def run_backoff_optimization(ev, x0, bounds, margin_pct=0.05):
    """
    Simple back-off: tighten each constraint by a fixed percentage.

    margin_pct = 0.05 means 5% safety margin:
      LPG_C2 <= 12 * 0.95 = 11.4
      LPG_C5 <= 2  * 0.95 = 1.9
      NG_RVP <= 76 * 0.95 = 72.2
      SG_C1  >= 80 * 1.05 = 84.0
    """
    print(f"\n\n{'#' * 60}")
    print(f"APPROACH 1: BACK-OFF ({margin_pct*100:.0f}% safety margin)")
    print(f"{'#' * 60}")

    constraints_backoff = [
        NonlinearConstraint(ev.g1, LIM_G1 * (1 + margin_pct), np.inf),
        NonlinearConstraint(ev.g2, 0.0, LIM_G2 * (1 - margin_pct)),
        NonlinearConstraint(ev.g3, 0.0, LIM_G3 * (1 - margin_pct)),
        NonlinearConstraint(ev.g4, 0.0, LIM_G4 * (1 - margin_pct)),
    ]

    ev.reset_counter()
    ev.cache.clear()

    start = time.time()
    res = minimize(
        ev.objective,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints_backoff,
        options={'maxiter': 100, 'ftol': 1e-2, 'eps': 0.5}
    )
    elapsed = time.time() - start

    print(f"\n  Optimizer status: {res.message}")
    print(f"  DWSIM evaluations: {ev.n_evaluations}")
    rev, min_margin = print_solution(
        f"Back-off {margin_pct*100:.0f}% — Result", res.x, ev, elapsed)

    return res, rev, min_margin


# ============================================================
# APPROACH 2: PENALISED OBJECTIVE (Revenue × Robustness trade-off)
# ============================================================

def run_penalised_optimization(ev, x0, bounds, alpha=0.1):
    """
    Penalise the objective by inverse distance to the closest constraint.

    alpha controls the weight:
      alpha = 0    → pure revenue (recovers original SLSQP)
      alpha = 0.1  → mild robustness preference
      alpha = 1.0  → strong robustness preference
    """
    print(f"\n\n{'#' * 60}")
    print(f"APPROACH 2: PENALISED OBJECTIVE (alpha = {alpha})")
    print(f"{'#' * 60}")

    def penalised_objective(x):
        rev = ev.objective(x)  # negative (minimisation)

        g2 = ev.g2(x)
        g3 = ev.g3(x)
        g4 = ev.g4(x)

        margin_g2 = (LIM_G2 - g2) / LIM_G2
        margin_g3 = (LIM_G3 - g3) / LIM_G3
        margin_g4 = (LIM_G4 - g4) / LIM_G4

        min_margin = min(margin_g2, margin_g3, margin_g4)

        # Barrier-like penalty: grows as margin → 0
        # Clamped to avoid division by zero
        safe_margin = max(min_margin, 0.001)
        penalty = alpha * abs(rev) * (1.0 / safe_margin - 1.0)

        return rev + penalty

    constraints_original = [
        NonlinearConstraint(ev.g1, LIM_G1, np.inf),
        NonlinearConstraint(ev.g2, 0.0, LIM_G2),
        NonlinearConstraint(ev.g3, 0.0, LIM_G3),
        NonlinearConstraint(ev.g4, 0.0, LIM_G4),
    ]

    ev.reset_counter()
    ev.cache.clear()

    start = time.time()
    res = minimize(
        penalised_objective,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints_original,
        options={'maxiter': 100, 'ftol': 1e-2, 'eps': 0.5}
    )
    elapsed = time.time() - start

    print(f"\n  Optimizer status: {res.message}")
    print(f"  DWSIM evaluations: {ev.n_evaluations}")
    rev, min_margin = print_solution(
        f"Penalised (alpha={alpha}) — Result", res.x, ev, elapsed)

    return res, rev, min_margin


# ============================================================
# APPROACH 3: TWO-STAGE (Maximise min-margin subject to min revenue)
# ============================================================

def run_twostage_optimization(ev, x0, bounds, rev_min_fraction=0.95):
    """
    Stage 1: Already done (SLSQP → 663.85 $/h)
    Stage 2: Maximise the minimum normalised constraint margin,
             subject to Revenue >= rev_min_fraction * REV_OPTIMAL.

    This finds the point FARTHEST from all constraint boundaries
    while still achieving acceptable revenue.
    """
    rev_min = REV_OPTIMAL * rev_min_fraction

    print(f"\n\n{'#' * 60}")
    print(f"APPROACH 3: TWO-STAGE (min revenue = {rev_min:.2f} $/h, "
          f"{rev_min_fraction*100:.0f}% of optimal)")
    print(f"{'#' * 60}")

    def min_margin_objective(x):
        """Minimise the negative of the smallest margin → maximise smallest margin."""
        g2 = ev.g2(x)
        g3 = ev.g3(x)
        g4 = ev.g4(x)

        margin_g2 = (LIM_G2 - g2) / LIM_G2
        margin_g3 = (LIM_G3 - g3) / LIM_G3
        margin_g4 = (LIM_G4 - g4) / LIM_G4

        return -min(margin_g2, margin_g3, margin_g4)

    def revenue_constraint(x):
        """Revenue must be >= rev_min."""
        return -ev.objective(x)  # objective returns negative → negate

    constraints_stage2 = [
        NonlinearConstraint(ev.g1, LIM_G1, np.inf),
        NonlinearConstraint(ev.g2, 0.0, LIM_G2),
        NonlinearConstraint(ev.g3, 0.0, LIM_G3),
        NonlinearConstraint(ev.g4, 0.0, LIM_G4),
        NonlinearConstraint(revenue_constraint, rev_min, np.inf),
    ]

    ev.reset_counter()
    ev.cache.clear()

    start = time.time()
    res = minimize(
        min_margin_objective,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints_stage2,
        options={'maxiter': 100, 'ftol': 1e-3, 'eps': 0.5}
    )
    elapsed = time.time() - start

    print(f"\n  Optimizer status: {res.message}")
    print(f"  DWSIM evaluations: {ev.n_evaluations}")
    rev, min_margin = print_solution(
        f"Two-stage ({rev_min_fraction*100:.0f}% revenue) — Result",
        res.x, ev, elapsed)

    return res, rev, min_margin


In [39]:
# ============================================================
# RUN ALL APPROACHES
# ============================================================
# Assumes ev, x0, bounds are already defined in the notebook

print("\n" + "=" * 60)
print("ROBUST OPTIMIZATION ANALYSIS")
print("=" * 60)

# ── 1. Original SLSQP (for reference) ──
print("\n\n" + "=" * 60)
print("REFERENCE: Original SLSQP (boundary-hugging)")
print("=" * 60)

ev.reset_counter()
ev.cache.clear()

constraints_original = [
    NonlinearConstraint(ev.g1, LIM_G1, np.inf),
    NonlinearConstraint(ev.g2, 0.0, LIM_G2),
    NonlinearConstraint(ev.g3, 0.0, LIM_G3),
    NonlinearConstraint(ev.g4, 0.0, LIM_G4),
]

start = time.time()
res_original = minimize(
    ev.objective, x0, method='SLSQP', bounds=bounds,
    constraints=constraints_original,
    options={'maxiter': 100, 'ftol': 1e-2, 'eps': 0.5}
)
time_original = time.time() - start
rev_orig, margin_orig = print_solution(
    "Original SLSQP", res_original.x, ev, time_original)




ROBUST OPTIMIZATION ANALYSIS


REFERENCE: Original SLSQP (boundary-hugging)

  Original SLSQP
  Revenue = 663.85 $/h
  Min normalised margin = -0.0000 (-0.00%)
  Elapsed = 218.1 s

  Decision variables:
    V_02_Temp      =   -33.0000
    T_01_Reb       =    63.5282
    T_02_RR        =     4.3119
    T_02_Reb       =    15.9579
    T_03_Reb       =    48.0000

  Constraints:
    Constraint        Value      Limit     Margin Status
    --------------------------------------------------------
    SG_C1           83.7441    >= 80.0     +4.68% ✓
    LPG_C2          11.9999    <= 12.0     +0.00% ✓
    LPG_C5           0.0056     <= 2.0    +99.72% ✓
    NG_RVP          76.0021    <= 76.0     -0.00% ✗ VIOLATED


In [40]:
# ── 2. Back-off approaches ──
res_bo1, rev_bo1, margin_bo1 = run_backoff_optimization(
    ev, res_original.x, bounds, margin_pct=0.01)
res_bo2, rev_bo2, margin_bo2 = run_backoff_optimization(
    ev, res_original.x, bounds, margin_pct=0.02)
res_bo3, rev_bo3, margin_bo3 = run_backoff_optimization(
    ev, res_original.x, bounds, margin_pct=0.03)



############################################################
APPROACH 1: BACK-OFF (1% safety margin)
############################################################

  Optimizer status: Optimization terminated successfully
  DWSIM evaluations: 14

  Back-off 1% — Result
  Revenue = 663.76 $/h
  Min normalised margin = 0.0100 (1.00%)
  Elapsed = 87.3 s

  Decision variables:
    V_02_Temp      =   -33.0000
    T_01_Reb       =    63.3824
    T_02_RR        =     4.3146
    T_02_Reb       =    15.9062
    T_03_Reb       =    47.9648

  Constraints:
    Constraint        Value      Limit     Margin Status
    --------------------------------------------------------
    SG_C1           83.7368    >= 80.0     +4.67% ✓
    LPG_C2          11.8795    <= 12.0     +1.00% ✓
    LPG_C5           0.0057     <= 2.0    +99.72% ✓
    NG_RVP          75.2309    <= 76.0     +1.01% ✓


############################################################
APPROACH 1: BACK-OFF (2% safety margin)
###################

In [41]:
# ── 3. Penalised objective ──
# With max joint margin of ~4.5%, use gentle alpha values
res_pen01, rev_pen01, margin_pen01 = run_penalised_optimization(
    ev, res_original.x, bounds, alpha=0.01)
res_pen05, rev_pen05, margin_pen05 = run_penalised_optimization(
    ev, res_original.x, bounds, alpha=0.05)
res_pen10, rev_pen10, margin_pen10 = run_penalised_optimization(
    ev, res_original.x, bounds, alpha=0.10)



############################################################
APPROACH 2: PENALISED OBJECTIVE (alpha = 0.01)
############################################################

  Optimizer status: Optimization terminated successfully
  DWSIM evaluations: 98

  Penalised (alpha=0.01) — Result
  Revenue = 651.17 $/h
  Min normalised margin = 0.0357 (3.57%)
  Elapsed = 618.0 s

  Decision variables:
    V_02_Temp      =   -28.9848
    T_01_Reb       =    61.4107
    T_02_RR        =     4.3526
    T_02_Reb       =    15.1325
    T_03_Reb       =    46.6831

  Constraints:
    Constraint        Value      Limit     Margin Status
    --------------------------------------------------------
    SG_C1           82.8555    >= 80.0     +3.57% ✓
    LPG_C2           7.0362    <= 12.0    +41.36% ✓
    LPG_C5           1.3253     <= 2.0    +33.74% ✓
    NG_RVP          50.3687    <= 76.0    +33.73% ✓


############################################################
APPROACH 2: PENALISED OBJECTIVE (alpha =

In [42]:
# ── 4. Two-stage optimizations ──
# Max joint margin is ~4.5%, so even 99% revenue is tight
res_ts99, rev_ts99, margin_ts99 = run_twostage_optimization(
    ev, res_original.x, bounds, rev_min_fraction=0.995)
res_ts98, rev_ts98, margin_ts98 = run_twostage_optimization(
    ev, res_original.x, bounds, rev_min_fraction=0.99)
res_ts95, rev_ts95, margin_ts95 = run_twostage_optimization(
    ev, res_original.x, bounds, rev_min_fraction=0.95)



############################################################
APPROACH 3: TWO-STAGE (min revenue = 660.53 $/h, 100% of optimal)
############################################################

  Optimizer status: Optimization terminated successfully
  DWSIM evaluations: 56

  Two-stage (100% revenue) — Result
  Revenue = 660.53 $/h
  Min normalised margin = 0.0440 (4.40%)
  Elapsed = 415.9 s

  Decision variables:
    V_02_Temp      =   -32.7782
    T_01_Reb       =    60.0000
    T_02_RR        =     4.3162
    T_02_Reb       =    13.4506
    T_03_Reb       =    47.3200

  Constraints:
    Constraint        Value      Limit     Margin Status
    --------------------------------------------------------
    SG_C1           83.5169    >= 80.0     +4.40% ✓
    LPG_C2           8.6773    <= 12.0    +27.69% ✓
    LPG_C5           1.2070     <= 2.0    +39.65% ✓
    NG_RVP          50.4992    <= 76.0    +33.55% ✓


############################################################
APPROACH 3: TWO-STA

## Generate Optimization Report

In [44]:
# ============================================================
# VISUALIZATION OF ROBUST OPTIMIZATION RESULTS
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ============================================================
# DATA FROM NOTEBOOK RESULTS (hardcoded from DWSIM outputs)
# ============================================================

results = {
    "SLSQP (original)": {
        "revenue": 663.85, "min_margin": -0.00, "elapsed": 218.1, "evals": 32,
        "dvs": {"V_02_Temp": -33.00, "T_01_Reb": 63.53, "T_02_RR": 4.31, "T_02_Reb": 15.96, "T_03_Reb": 48.00},
        "constraints": {"SG_C1": {"val": 83.74, "margin": 4.68}, "LPG_C2": {"val": 12.00, "margin": 0.00},
                        "LPG_C5": {"val": 0.01, "margin": 99.72}, "NG_RVP": {"val": 76.00, "margin": -0.00}},
    },
    "Back-off 1%": {
        "revenue": 663.76, "min_margin": 1.00, "elapsed": 87.3, "evals": 14,
        "dvs": {"V_02_Temp": -33.00, "T_01_Reb": 63.38, "T_02_RR": 4.31, "T_02_Reb": 15.91, "T_03_Reb": 47.96},
        "constraints": {"SG_C1": {"val": 83.74, "margin": 4.67}, "LPG_C2": {"val": 11.88, "margin": 1.00},
                        "LPG_C5": {"val": 0.01, "margin": 99.72}, "NG_RVP": {"val": 75.23, "margin": 1.01}},
    },
    "Back-off 2%": {
        "revenue": 663.66, "min_margin": 2.02, "elapsed": 88.2, "evals": 14,
        "dvs": {"V_02_Temp": -33.00, "T_01_Reb": 63.24, "T_02_RR": 4.31, "T_02_Reb": 15.85, "T_03_Reb": 47.93},
        "constraints": {"SG_C1": {"val": 83.73, "margin": 4.66}, "LPG_C2": {"val": 11.76, "margin": 2.02},
                        "LPG_C5": {"val": 0.01, "margin": 99.71}, "NG_RVP": {"val": 74.47, "margin": 2.02}},
    },
    "Back-off 3%": {
        "revenue": 663.56, "min_margin": 3.04, "elapsed": 90.1, "evals": 14,
        "dvs": {"V_02_Temp": -33.00, "T_01_Reb": 63.09, "T_02_RR": 4.31, "T_02_Reb": 15.80, "T_03_Reb": 47.90},
        "constraints": {"SG_C1": {"val": 83.72, "margin": 4.65}, "LPG_C2": {"val": 11.64, "margin": 3.04},
                        "LPG_C5": {"val": 0.01, "margin": 99.70}, "NG_RVP": {"val": 73.69, "margin": 3.05}},
    },
    "Penalised α=0.01": {
        "revenue": 651.17, "min_margin": 3.57, "elapsed": 618.0, "evals": 98,
        "dvs": {"V_02_Temp": -28.98, "T_01_Reb": 61.41, "T_02_RR": 4.35, "T_02_Reb": 15.13, "T_03_Reb": 46.68},
        "constraints": {"SG_C1": {"val": 82.86, "margin": 3.57}, "LPG_C2": {"val": 7.04, "margin": 41.36},
                        "LPG_C5": {"val": 1.33, "margin": 33.74}, "NG_RVP": {"val": 50.37, "margin": 33.73}},
    },
    "Penalised α=0.05": {
        "revenue": 651.51, "min_margin": 3.61, "elapsed": 649.0, "evals": 108,
        "dvs": {"V_02_Temp": -28.99, "T_01_Reb": 62.05, "T_02_RR": 4.36, "T_02_Reb": 14.89, "T_03_Reb": 47.13},
        "constraints": {"SG_C1": {"val": 82.89, "margin": 3.61}, "LPG_C2": {"val": 7.63, "margin": 36.42},
                        "LPG_C5": {"val": 1.32, "margin": 33.76}, "NG_RVP": {"val": 50.36, "margin": 33.74}},
    },
    "Penalised α=0.10": {
        "revenue": 651.50, "min_margin": 3.61, "elapsed": 596.4, "evals": 108,
        "dvs": {"V_02_Temp": -28.99, "T_01_Reb": 62.04, "T_02_RR": 4.33, "T_02_Reb": 14.94, "T_03_Reb": 47.01},
        "constraints": {"SG_C1": {"val": 82.89, "margin": 3.61}, "LPG_C2": {"val": 7.61, "margin": 36.60},
                        "LPG_C5": {"val": 1.32, "margin": 33.83}, "NG_RVP": {"val": 50.36, "margin": 33.74}},
    },
    "Two-stage 99.5%": {
        "revenue": 660.53, "min_margin": 4.40, "elapsed": 415.9, "evals": 56,
        "dvs": {"V_02_Temp": -32.78, "T_01_Reb": 60.00, "T_02_RR": 4.32, "T_02_Reb": 13.45, "T_03_Reb": 47.32},
        "constraints": {"SG_C1": {"val": 83.52, "margin": 4.40}, "LPG_C2": {"val": 8.68, "margin": 27.69},
                        "LPG_C5": {"val": 1.21, "margin": 39.65}, "NG_RVP": {"val": 50.50, "margin": 33.55}},
    },
    "Two-stage 99.0%": {
        "revenue": 657.89, "min_margin": 4.16, "elapsed": 849.6, "evals": 112,
        "dvs": {"V_02_Temp": -31.77, "T_01_Reb": 60.00, "T_02_RR": 4.32, "T_02_Reb": 13.85, "T_03_Reb": 47.41},
        "constraints": {"SG_C1": {"val": 83.33, "margin": 4.16}, "LPG_C2": {"val": 7.96, "margin": 33.70},
                        "LPG_C5": {"val": 1.21, "margin": 39.59}, "NG_RVP": {"val": 50.57, "margin": 33.46}},
    },
    "Two-stage 95.0%": {
        "revenue": 656.46, "min_margin": 4.03, "elapsed": 785.9, "evals": 96,
        "dvs": {"V_02_Temp": -31.19, "T_01_Reb": 60.07, "T_02_RR": 4.33, "T_02_Reb": 14.00, "T_03_Reb": 47.51},
        "constraints": {"SG_C1": {"val": 83.22, "margin": 4.03}, "LPG_C2": {"val": 7.62, "margin": 36.53},
                        "LPG_C5": {"val": 1.28, "margin": 35.97}, "NG_RVP": {"val": 50.41, "margin": 33.67}},
    },
}

names = list(results.keys())
revs = [results[n]["revenue"] for n in names]
margins = [results[n]["min_margin"] for n in names]

# Approach categories for coloring
colors_map = {
    "SLSQP (original)": "#d62728",
    "Back-off 1%": "#1f77b4", "Back-off 2%": "#1f77b4", "Back-off 3%": "#1f77b4",
    "Penalised α=0.01": "#9467bd", "Penalised α=0.05": "#9467bd", "Penalised α=0.10": "#9467bd",
    "Two-stage 99.5%": "#2ca02c", "Two-stage 99.0%": "#2ca02c", "Two-stage 95.0%": "#2ca02c",
}
colors = [colors_map[n] for n in names]


# ──────────────────────────────────────────────────────────────
# FIGURE 1: Revenue vs Minimum Margin (scatter)
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5.5))

for i, n in enumerate(names):
    marker = {"SLSQP (original)": "*"}.get(n, "o")
    size = 200 if n == "SLSQP (original)" else 80
    zorder = 10 if n == "SLSQP (original)" else 5
    ax.scatter(margins[i], revs[i], c=colors[i], s=size, marker=marker,
               edgecolors="black", linewidths=0.6, zorder=zorder)
    # Label
    offset_x, offset_y = 0.15, 0.0
    if n == "SLSQP (original)":
        offset_x, offset_y = 0.15, 1.2
    elif "Two-stage" in n:
        offset_y = -1.5
    elif "Penalised" in n:
        offset_y = 1.0
    ax.annotate(n, (margins[i], revs[i]),
                xytext=(margins[i] + offset_x, revs[i] + offset_y),
                fontsize=7, ha="left",
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.5) if abs(offset_y) > 0.5 else None)

# Legend patches
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="*", color="w", markerfacecolor="#d62728",
           markeredgecolor="black", markersize=12, label="Original SLSQP"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#1f77b4",
           markeredgecolor="black", markersize=8, label="Back-off"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#9467bd",
           markeredgecolor="black", markersize=8, label="Penalised objective"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2ca02c",
           markeredgecolor="black", markersize=8, label="Two-stage (max margin)"),
]
ax.legend(handles=legend_elements, loc="center left", framealpha=0.9)

ax.set_xlabel("Minimum normalised constraint margin (%)")
ax.set_ylabel("Revenue ($/h)")
ax.set_title("Revenue vs Operational Robustness — All Approaches", fontweight="bold")
ax.axvline(0, color="red", ls=":", lw=1, alpha=0.4)
ax.text(0.1, ax.get_ylim()[0] + 0.5, "boundary", fontsize=7, color="red", alpha=0.6)

plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig_revenue_vs_margin.png")
plt.savefig(f"{FIGDIR}/fig_revenue_vs_margin.pdf")
plt.close()
print("✓ Figure 1: Revenue vs Margin")


# ──────────────────────────────────────────────────────────────
# FIGURE 2: Constraint margin breakdown (grouped bar)
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

constraint_names = ["SG_C1", "LPG_C2", "NG_RVP"]  # skip LPG_C5 (always ~99%)
bar_colors = {"SG_C1": "#e74c3c", "LPG_C2": "#3498db", "NG_RVP": "#f39c12"}

x_pos = np.arange(len(names))
width = 0.25
offsets = [-width, 0, width]

for j, cname in enumerate(constraint_names):
    vals = [results[n]["constraints"][cname]["margin"] for n in names]
    # Cap at 50% for readability
    vals_capped = [min(v, 50) for v in vals]
    bars = ax.bar(x_pos + offsets[j], vals_capped, width, label=cname,
                  color=bar_colors[cname], alpha=0.85, edgecolor="white", linewidth=0.5)
    # Add value labels on top
    for k, (bar, val) in enumerate(zip(bars, vals)):
        if val > 0.5:
            label = f"{val:.1f}" if val < 50 else f"{val:.0f}"
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    label, ha="center", va="bottom", fontsize=6.5)

ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Constraint margin (%)")
ax.set_title("Individual Constraint Margins by Approach", fontweight="bold")
ax.legend(loc="upper right", framealpha=0.9)
ax.set_ylim(0, 55)
ax.axhline(5, color="gray", ls=":", lw=0.8, alpha=0.5)
ax.text(len(names)-0.5, 5.5, "5% margin", fontsize=7, color="gray", ha="right")

plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig_margin_breakdown.png")
plt.savefig(f"{FIGDIR}/fig_margin_breakdown.pdf")
plt.close()
print("✓ Figure 2: Constraint margin breakdown")


# ──────────────────────────────────────────────────────────────
# FIGURE 3: Decision variables comparison (parallel-like)
# ──────────────────────────────────────────────────────────────
dv_names = ["V_02_Temp", "T_01_Reb", "T_02_RR", "T_02_Reb", "T_03_Reb"]
dv_labels = ["V-02 Temp\n(°C)", "T-01 Reb\n(°C)", "T-02 RR\n(—)", "T-02 Reb\n(°C)", "T-03 Reb\n(°C)"]
dv_bounds = [(-33, -17), (60, 77), (0.7, 4.5), (9, 27), (45, 64)]

# Select representative approaches
selected = ["SLSQP (original)", "Back-off 3%", "Penalised α=0.05", "Two-stage 99.5%"]
sel_colors = ["#d62728", "#1f77b4", "#9467bd", "#2ca02c"]
sel_markers = ["*", "s", "D", "o"]

fig, axes = plt.subplots(1, 5, figsize=(14, 4), sharey=False)

for j, (dv, label, (lb, ub)) in enumerate(zip(dv_names, dv_labels, dv_bounds)):
    ax = axes[j]
    for i, (name, color, marker) in enumerate(zip(selected, sel_colors, sel_markers)):
        val = results[name]["dvs"][dv]
        ms = 12 if name == "SLSQP (original)" else 8
        ax.scatter(i, val, c=color, marker=marker, s=ms**2,
                   edgecolors="black", linewidths=0.5, zorder=5)
    ax.set_ylim(lb - (ub-lb)*0.05, ub + (ub-lb)*0.05)
    ax.axhline(lb, color="gray", ls=":", lw=0.8, alpha=0.5)
    ax.axhline(ub, color="gray", ls=":", lw=0.8, alpha=0.5)
    ax.fill_between([-0.5, len(selected)-0.5], lb, ub, alpha=0.04, color="blue")
    ax.set_xticks(range(len(selected)))
    ax.set_xticklabels([])
    ax.set_xlabel(label, fontsize=9)
    if j == 0:
        ax.set_ylabel("Value")

# Custom legend below
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker=m, color="w", markerfacecolor=c,
           markeredgecolor="black", markersize=9, label=n)
    for n, c, m in zip(selected, sel_colors, sel_markers)
]
fig.legend(handles=legend_elements, loc="lower center", ncol=4,
           framealpha=0.9, fontsize=8.5, bbox_to_anchor=(0.5, -0.02))
plt.suptitle("Decision Variables — Selected Approaches", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig_decision_variables.png")
plt.savefig(f"{FIGDIR}/fig_decision_variables.pdf")
plt.close()
print("✓ Figure 3: Decision variables comparison")


# ──────────────────────────────────────────────────────────────
# FIGURE 4: Revenue loss vs Margin gain (trade-off bar chart)
# ──────────────────────────────────────────────────────────────
ref_rev = results["SLSQP (original)"]["revenue"]

fig, ax1 = plt.subplots(figsize=(10, 5))

approaches = names[1:]  # skip original
rev_loss = [ref_rev - results[n]["revenue"] for n in approaches]
margin_gain = [results[n]["min_margin"] for n in approaches]
bar_c = [colors_map[n] for n in approaches]

x_pos = np.arange(len(approaches))

bars1 = ax1.bar(x_pos - 0.18, rev_loss, 0.35, color=[c + "AA" for c in bar_c],
                edgecolor="black", linewidth=0.3, label="Revenue loss ($/h)")
ax1.set_ylabel("Revenue loss vs SLSQP ($/h)", color="#333")
ax1.set_ylim(0, max(rev_loss) * 1.3)

ax2 = ax1.twinx()
bars2 = ax2.bar(x_pos + 0.18, margin_gain, 0.35, color=bar_c,
                edgecolor="black", linewidth=0.3, alpha=0.7, label="Min margin (%)")
ax2.set_ylabel("Minimum constraint margin (%)", color="#333")
ax2.set_ylim(0, max(margin_gain) * 1.3)

# Add value labels
for bar, val in zip(bars1, rev_loss):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
             f"{val:.2f}", ha="center", va="bottom", fontsize=7)
for bar, val in zip(bars2, margin_gain):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f"{val:.2f}%", ha="center", va="bottom", fontsize=7)

ax1.set_xticks(x_pos)
ax1.set_xticklabels(approaches, rotation=35, ha="right", fontsize=8)
ax1.set_title("Trade-off: Revenue Loss vs Margin Gain", fontweight="bold")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", framealpha=0.9)

plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig_tradeoff_bars.png")
plt.savefig(f"{FIGDIR}/fig_tradeoff_bars.pdf")
plt.close()
print("✓ Figure 4: Trade-off bars")


# ──────────────────────────────────────────────────────────────
# FIGURE 5: Spider/Radar chart — constraint margins for 4 key approaches
# ──────────────────────────────────────────────────────────────
constraint_labels = ["SG_C1\n(≥ 80)", "LPG_C2\n(≤ 12)", "LPG_C5\n(≤ 2)", "NG_RVP\n(≤ 76)"]
angles = np.linspace(0, 2 * np.pi, 4, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for name, color, ls in zip(selected, sel_colors, ["-", "--", "-.", ":"]):
    vals = [results[name]["constraints"][c]["margin"] for c in ["SG_C1", "LPG_C2", "LPG_C5", "NG_RVP"]]
    # Cap at 40 for visualization
    vals_capped = [min(v, 40) for v in vals]
    vals_plot = vals_capped + vals_capped[:1]
    ax.plot(angles, vals_plot, color=color, linewidth=2, linestyle=ls, label=name)
    ax.fill(angles, vals_plot, color=color, alpha=0.06)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(constraint_labels, fontsize=9)
ax.set_ylim(0, 42)
ax.set_yticks([5, 10, 20, 30, 40])
ax.set_yticklabels(["5%", "10%", "20%", "30%", "40%"], fontsize=7, color="gray")
ax.set_title("Constraint Margin Profiles", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), framealpha=0.9, fontsize=8)

plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig_radar_margins.png")
plt.savefig(f"{FIGDIR}/fig_radar_margins.pdf")
plt.close()
print("✓ Figure 5: Radar chart")


# ──────────────────────────────────────────────────────────────
# TABLE 1: Summary comparison (save as CSV)
# ──────────────────────────────────────────────────────────────
table_rows = []
for name in names:
    r = results[name]
    table_rows.append({
        "Approach": name,
        "Revenue ($/h)": r["revenue"],
        "Rev. loss ($/h)": round(ref_rev - r["revenue"], 2),
        "Min margin (%)": r["min_margin"],
        "SG_C1 margin (%)": r["constraints"]["SG_C1"]["margin"],
        "LPG_C2 margin (%)": r["constraints"]["LPG_C2"]["margin"],
        "LPG_C5 margin (%)": r["constraints"]["LPG_C5"]["margin"],
        "NG_RVP margin (%)": r["constraints"]["NG_RVP"]["margin"],
        "DWSIM evals": r["evals"],
        "Time (s)": r["elapsed"],
    })

df_summary = pd.DataFrame(table_rows)
df_summary.to_csv(f"{FIGDIR}/table_summary_comparison.csv", index=False)
print(f"\n✓ Table saved: {FIGDIR}/table_summary_comparison.csv")
print("\n" + "=" * 90)
print(df_summary.to_string(index=False))
print("=" * 90)


# ──────────────────────────────────────────────────────────────
# TABLE 2: Decision variables at each approach (save as CSV)
# ──────────────────────────────────────────────────────────────
dv_rows = []
for name in names:
    row = {"Approach": name, "Revenue ($/h)": results[name]["revenue"]}
    row.update(results[name]["dvs"])
    dv_rows.append(row)

df_dvs = pd.DataFrame(dv_rows)
df_dvs.to_csv(f"{FIGDIR}/table_decision_variables.csv", index=False)
print(f"✓ Table saved: {FIGDIR}/table_decision_variables.csv")


# ──────────────────────────────────────────────────────────────
# FIGURE 6: Efficiency metric — Revenue per % margin
# ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))

# Efficiency = revenue loss per % of margin gained
approaches_eff = names[1:]
eff = []
for n in approaches_eff:
    loss = ref_rev - results[n]["revenue"]
    marg = results[n]["min_margin"]
    eff.append(loss / marg if marg > 0 else 0)

bar_c_eff = [colors_map[n] for n in approaches_eff]
x_pos = np.arange(len(approaches_eff))
bars = ax.bar(x_pos, eff, color=bar_c_eff, edgecolor="black", linewidth=0.3, alpha=0.85)

for bar, val in zip(bars, eff):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x_pos)
ax.set_xticklabels(approaches_eff, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Revenue cost per 1% margin ($/h per %)")
ax.set_title("Efficiency: Revenue Cost per Unit of Margin Gained", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig_efficiency.png")
plt.savefig(f"{FIGDIR}/fig_efficiency.pdf")
plt.close()
print("✓ Figure 6: Efficiency metric")

print(f"\n{'='*60}")
print(f"All figures and tables saved to: {FIGDIR}/")
print(f"{'='*60}")

✓ Figure 1: Revenue vs Margin
✓ Figure 2: Constraint margin breakdown
✓ Figure 3: Decision variables comparison
✓ Figure 4: Trade-off bars
✓ Figure 5: Radar chart

✓ Table saved: operability/optimization_figures/table_summary_comparison.csv

        Approach  Revenue ($/h)  Rev. loss ($/h)  Min margin (%)  SG_C1 margin (%)  LPG_C2 margin (%)  LPG_C5 margin (%)  NG_RVP margin (%)  DWSIM evals  Time (s)
SLSQP (original)         663.85             0.00           -0.00              4.68               0.00              99.72              -0.00           32     218.1
     Back-off 1%         663.76             0.09            1.00              4.67               1.00              99.72               1.01           14      87.3
     Back-off 2%         663.66             0.19            2.02              4.66               2.02              99.71               2.02           14      88.2
     Back-off 3%         663.56             0.29            3.04              4.65               3.04     